# 01. Data Loading and Exploratory Data Analysis (EDA)

## 1. Introduction
This notebook examines raw backlink pricing data to understand the market dynamics of link placements. The goal is to prepare a clean, well-understood dataset for building a price prediction model based on domain quality metrics.

## 2. Objectives

**O1. Data understanding and hygiene**
Load raw data from Supabase extract; examine missingness, duplicates, and data types; filter to valid price ranges; validate metric bounds.

**O2. Feature derivation**
Extract TLD from domain names; normalize country codes and categorical fields; compute log transforms for skewed distributions.

**O3. Distribution analysis**
Visualize price, quality metrics (DR, CF, TF), traffic, TLD, and country distributions to identify patterns and potential modeling challenges.

## 3. Sections
| # | Section | Purpose |
|---|---------|--------|
| 4 | Environment setup | Imports, logging, paths |
| 5 | Data loading | Load raw Parquet from extraction pipeline |
| 6 | Missing value analysis | Quantify and visualize gaps |
| 7 | Data cleaning | Price filter, metric validation, normalization |
| 8 | Price distribution | Raw and log-transformed price histograms |
| 9 | Quality metric distributions | DR, CF, TF histograms and price-by-tier |
| 10 | TLD analysis | TLD extraction, distribution, price by TLD |
| 11 | Country and source analysis | Geographic and acquisition channel breakdown |
| 12 | Summary | Key findings and output artifacts |

In [ ]:
import subprocess
import sys
from pathlib import Path


def _ensure_local_repo_src_on_path() -> None:
    for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
        package_root = candidate / "backlink_pricing_model"
        if package_root.exists():
            candidate_str = str(candidate.resolve())
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return


_ensure_local_repo_src_on_path()

In [ ]:
import logging
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

from backlink_pricing_model.core.environment import get_project_root
from backlink_pricing_model.preprocessing.data_loading import (
    load_raw_parquet,
    save_processed,
)
from backlink_pricing_model.preprocessing.data_quality import (
    detect_outliers_iqr,
    filter_valid_prices,
    missing_value_report,
    validate_metric_ranges,
)
from backlink_pricing_model.preprocessing.feature_engineering import (
    add_log_price,
    add_log_traffic,
    add_tld_feature,
    normalize_country,
    normalize_link_source_type,
    normalize_link_type,
)
from backlink_pricing_model.visualization.distributions_plots import (
    plot_country_distribution,
    plot_metric_distributions,
    plot_missing_values,
    plot_price_by_quality_tier,
    plot_price_by_tld,
    plot_price_distribution,
    plot_tld_distribution,
)
from backlink_pricing_model.visualization.plots_style import save_plot

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = get_project_root()
EDA_IMAGES_DIR = PROJECT_ROOT / "images" / "eda"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EDA_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Project root: %s", PROJECT_ROOT)

## 5. Data loading

Load the raw backlink dataset extracted from Supabase via `make extract`. The loader logs row counts and column types.

In [ ]:
df_raw = load_raw_parquet()
logger.info("Loaded %d rows, %d columns", len(df_raw), len(df_raw.columns))
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

## 6. Missing value analysis

In [ ]:
report = missing_value_report(df_raw)
display(report)

In [ ]:
fig = plot_missing_values(df_raw)
save_plot(fig, "missing_values", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "missing_values.png"))

_Figure 1. Percentage of missing values per column. CF and TF have the highest missingness (~66%), while DR and traffic are nearly complete._

## 7. Data cleaning

Filter to valid price range, validate quality metric bounds, normalize categorical fields, and extract TLD from domains.

In [ ]:
df = filter_valid_prices(df_raw, min_price=0.0, max_price=50_000.0)
df = validate_metric_ranges(df)

df = normalize_country(df)
df = normalize_link_type(df)
df = normalize_link_source_type(df)

df = add_tld_feature(df)
df = add_log_price(df)
df = add_log_traffic(df)

logger.info("Clean dataset: %d rows, %d columns", len(df), len(df.columns))
df.describe()

## 8. Price distribution

In [ ]:
fig = plot_price_distribution(df, log_scale=True)
save_plot(fig, "price_distribution_log", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "price_distribution_log.png"))

_Figure 2. Log-transformed price distribution. The distribution is approximately normal after log transform, supporting the use of log_price as the target variable._

In [ ]:
fig = plot_price_distribution(df, log_scale=False)
save_plot(fig, "price_distribution_raw", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "price_distribution_raw.png"))

_Figure 3. Raw price distribution. Heavily right-skewed with most placements under $500._

## 9. Quality metric distributions

In [ ]:
fig = plot_metric_distributions(df)
save_plot(fig, "quality_metric_distributions", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "quality_metric_distributions.png"))

_Figure 4. Distribution of DR, CF, and TF quality metrics. DR is roughly normal around 50, while CF and TF are right-skewed with lower median values._

In [ ]:
for metric in ["dr", "cf", "tf"]:
    fig = plot_price_by_quality_tier(df, metric=metric)
    save_plot(fig, f"price_by_{metric}_tier", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "price_by_dr_tier.png"))

_Figure 5. Price by DR tier. Clear monotonic relationship: higher DR domains command higher prices._

## 10. TLD analysis

In [ ]:
fig = plot_tld_distribution(df, top_n=15)
save_plot(fig, "tld_distribution", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "tld_distribution.png"))

_Figure 6. Top 15 TLDs by listing count. .com dominates, followed by country-code TLDs._

In [ ]:
fig = plot_price_by_tld(df, top_n=10)
save_plot(fig, "price_by_tld", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "price_by_tld.png"))

_Figure 7. Median price by TLD. TLD carries pricing signal beyond what DR alone captures._

## 11. Country and source analysis

In [ ]:
fig = plot_country_distribution(df, top_n=15)
save_plot(fig, "country_distribution", str(EDA_IMAGES_DIR))

In [ ]:
Image(filename=str(EDA_IMAGES_DIR / "country_distribution.png"))

_Figure 8. Top 15 countries by listing count. US dominates the dataset._

In [ ]:
source_stats = (
    df.groupby("link_source_type")["final_price"]
    .agg(["count", "mean", "median"])
    .round(2)
    .sort_values("count", ascending=False)
)
logger.info("Price stats by link source type:")
display(source_stats.head(10))

In [ ]:
price_outliers = detect_outliers_iqr(df["final_price"].dropna())
logger.info(
    "Price outliers (IQR): %d of %d (%.1f%%)",
    price_outliers.sum(),
    len(price_outliers),
    price_outliers.sum() / len(price_outliers) * 100,
)

---

## 12. Summary

This notebook audited the raw backlink dataset, cleaned it, and documented the main signals carried forward to feature engineering.

**Key findings:**
- **Price distribution:** Heavily right-skewed; log transform produces approximately normal distribution suitable for regression.
- **Quality metrics:** DR is the strongest single predictor of price (clear monotonic relationship). CF/TF have ~66% missingness but carry signal where available.
- **TLD signal:** Domain extension carries pricing signal beyond what quality metrics alone capture.
- **Country bias:** US dominates (~50%); country normalization maps inconsistent labels to ISO codes.
- **Data quality:** ~35K rows, ~34K with valid prices after filtering.

**Output artifacts:**
- `data/processed/backlinks_cleaned.parquet` — cleaned dataset for notebook 02
- `images/eda/` — all saved figures

In [ ]:
output_path = save_processed(df, "backlinks_cleaned", subdir="processed")
logger.info("Saved cleaned dataset (%d rows) to %s", len(df), output_path)